In [3]:
"""
    Create a MLflow experiment
    
    default experiment name is "Default"
"""
import mlflow
mlflow.set_tracking_uri(
    "azureml://canadacentral.api.azureml.ms/mlflow/v1.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice"
)
mlflow.set_experiment(experiment_name="heart-condition-classifier")


2026/10/08 19:44:24 INFO mlflow.tracking.fluent: Experiment with name 'heart-condition-classifier' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1791503065317, effective_trace_archival_retention=None, experiment_id='70d9617c-a3a1-4dfe-9ed0-da14bba30864', last_update_time=None, lifecycle_stage='active', name='heart-condition-classifier', tags={}, trace_location=None, workspace='default'>

In [4]:
"""
    Log results with MLflow
    
    Use start_run() to start a run tracked by MLflow.
    Next, to track the model, can:
    - enable autologging
    - use custom logging   
"""

'\n    Log results with MLflow\n\n    Use start_run() to start a run tracked by MLflow.\n    Next, to track the model, can:\n    - enable autologging\n    - use custom logging   \n'

In [5]:
"""
    Enable autologging
    
    MLflow supports automatic logging for popular machine learning libaries. When you enable autologging,
    MLflow instructs your framework to log metrics, parameters, artifacts, and models automatically.
    No need to specify what to log, MLflow will decide what's relevant.
    
    can turn on autologging by calling mlflow.autolog() before your training code.
    can also use framework-specific autologging APIs, such as mlflow.sklearn.autolog() or mlflow.pytorch.autolog().
"""

from xgboost import XGBClassifier

In [6]:
"""
    Prepare data: heart disease dataset, split into train / test
"""
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../cli/endpoints/batch/deploy-pipelines/training-with-components/data/train/heart.csv")

# thal 是字符串类别（normal / fixed / reversible），XGBoost 要求数值，先转成编码
df["thal"] = df["thal"].astype("category").cat.codes

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)


(212, 13) (91, 13)


In [10]:
with mlflow.start_run() as run:
    mlflow.autolog()

    model = XGBClassifier(eval_metric="logloss", early_stopping_rounds=10)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

wsid = "/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice"
print(f"https://ml.azure.com/runs/{run.info.run_id}?wsid={wsid}")

2026/10/08 19:58:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/08 19:58:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/10/08 19:59:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Yimin Chen\Desktop\azureml-examples\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#ha

🏃 View run careful_quince_h2ms3kq7 at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/70d9617c-a3a1-4dfe-9ed0-da14bba30864/runs/0c580ee7-c66e-4b3c-8c5a-844e27920b2c
🧪 View experiment at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/70d9617c-a3a1-4dfe-9ed0-da14bba30864
https://ml.azure.com/runs/0c580ee7-c66e-4b3c-8c5a-844e27920b2c?wsid=/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice


In [12]:
"""
    Use custom logging
    
    Additionally, you can manually log your model with MLflow.
    mlflow.log_params() and mlflow.log_metrics() can be used to log parameters and metrics, respectively.
    mlflow.log_figure() can be used to log figures, and mlflow.log_artifact() can be used to log artifacts.
    mlflow.log_model() can be used to log amodel.
    mlflow.log_image() can be used to log a numpy or PIL image as an artifact.
"""
from sklearn.metrics import accuracy_score
with mlflow.start_run() as run:
    mlflow.autolog()

    model = XGBClassifier(eval_metric="logloss", early_stopping_rounds=10)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

wsid = "/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice"
print(f"https://ml.azure.com/runs/{run.info.run_id}?wsid={wsid}")

2026/10/08 20:06:03 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/08 20:06:03 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/10/08 20:06:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Yimin Chen\Desktop\azureml-examples\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#ha

🏃 View run wheat_papaya_70x4dyf2 at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/70d9617c-a3a1-4dfe-9ed0-da14bba30864/runs/ff988eee-a5d8-4ac1-9a4e-633e8c68c57a
🧪 View experiment at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/70d9617c-a3a1-4dfe-9ed0-da14bba30864
https://ml.azure.com/runs/ff988eee-a5d8-4ac1-9a4e-633e8c68c57a?wsid=/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice


In [13]:
"""
    Evaluate models with the Responsible AI dashboard
    
    Why responsible AI matters
    
    MS 6 principles of responsible AI:
    1. Fairness: ensure model provides equitable outcomes by testing for and mitigating harmful bias across groups
    2. Reliability and safety: Build, test, and monitor model so it performs consistently and prevents unsafe behavior
    3. Privacy and security: Protect data and model from unauthorized access and use
    4. Inclusiveness: Ensure model is accessible and usable by people with diverse backgrounds, perspectives, and abilities
    5. Transparency: Provide clear information about model's purpose, capabilities, and limitations
    6. Accountability: Ensure people are accountable for the proper design, development, deployment, and use of AI systems    
"""

"\n    Evaluate models with the Responsible AI dashboard\n\n    Why responsible AI matters\n\n    MS 6 principles of responsible AI:\n    1. Fairness: ensure model provides equitable outcomes by testing for and mitigating harmful bias across groups\n    2. Reliability and safety: Build, test, and monitor model so it performs consistently and prevents unsafe behavior\n    3. Privacy and security: Protect data and model from unauthorized access and use\n    4. Inclusiveness: Ensure model is accessible and usable by people with diverse backgrounds, perspectives, and abilities\n    5. Transparency: Provide clear information about model's purpose, capabilities, and limitations\n    6. Accountability: Ensure people are accountable for the proper design, development, deployment, and use of AI systems    \n"

In [ ]:
"""
    Create a Responsible AI dashboard
"""